## A density based implementation of Schelling's Segregation Model using only: common language features and implementing roulette wheel selection

In [109]:
import random
import math
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import time

import svg_canvas

In [110]:
# Constants
MEAN_AGE = 40
STD_DEV_AGE = 15
MEAN_POP = 100
STD_DEV_POP = 10
GRID_SIZE = 20
SIMILARITY_THRESHOLD = 0.45
MAX_AGE = 120
MIN_AGE = 0
POP_MIX = 0.5
MAX_ITERATIONS = 20

## Create a population list

Create a 2D list where each row represents a single person, and each row contains the person's type, age, and which regional cell they occupy. This is identified by the cell's row and column.

In [111]:
def init_population(population, grid):
    c_max = len(grid)
    r_max = len(grid[0])
    pop_count = 0 # count number in population
    row = 0
    while row < r_max:   #row loop
        col = 0
        while col < c_max:   # col loop
            act_pop = int(random.gauss(MEAN_POP, STD_DEV_POP))
            person = 0
            while person < act_pop: # person loop
                age = random.gauss(MEAN_AGE, STD_DEV_AGE)
                if age < MIN_AGE:
                    age = MIN_AGE
                if age > MAX_AGE:
                    age = MAX_AGE
                age = int(age)
                
                if random.random() < POP_MIX:
                    person_type = 0
                else:
                    person_type = 1
                
                grid[row][col].append(pop_count)
                population.append([pop_count, person_type, age, row, col])
                pop_count = pop_count + 1
                
                person = person + 1  #person loop
            col = col + 1   # col loop
        row = row + 1   # row loop

## Create a spatial area
The spatial area is defined as a 3D list. The first two dimensions represent space, and the third dimension is a list of people from the population list, identified by their position in that list.

In [112]:
def init_grid(grid, similarity, distance, size):
    row = 0
    while row < size:
        grid.append([])
        similarity.append([])
        distance.append([])
        
        col = 0
        while col < size:
            grid[row].append([])
            similarity[row].append(0.0)
            
            # Initialise distance array 
            dist_row = []
            k = 0
            while k < size * size:
                dist_row.append(0.0)
                k = k + 1
            distance[row].append(dist_row)
            
            col = col + 1
        row = row + 1

# Calculate the homogeneity of each cell in the spatial grid

In [113]:
def calc_similarity(grid, population, similarity):
    r_max = len(grid)
    c_max = len(grid[0])
    
    row = 0
    while row < r_max:
        col = 0
        while col < c_max:
            # Reset counts for each cell
            count_t0 = 0
            count_t1 = 0
            
            pop = len(grid[row][col])
            p_idx = 0
            while p_idx < pop:
                p_id = grid[row][col][p_idx]
                if population[p_id][1] == 1:
                    count_t0 = count_t0 + 1
                else:
                    count_t1 = count_t1 + 1
                p_idx = p_idx + 1
            
            # Assign similarity - fixed index order (r,c)
            total = count_t0 + count_t1
            if total > 0:
                similarity[row][col] = float(count_t0) / float(total)
            else:
                similarity[row][col] = 0.0
                
            col = col + 1
        row = row + 1


In [114]:
def inverse_sqr_distance(x1, y1, x2, y2):
    dx = x2 - x1
    dy = y2 - y1
    dist_sq = dx * dx + dy * dy
    if dist_sq == 0:
        return 0.0
    return 1.0 / dist_sq

In [115]:
def move(y, x, grid, population, similarity):
    # Calculate roulette wheel weights
    r_max = len(grid)
    c_max = len(grid[0])
    total_weight = 0.0
    weights = []
    
    row = 0
    while row < r_max:
        col = 0
        while col < c_max:
            if y == row and x == col:
                d = 0.0
            else:
                d = inverse_sqr_distance(y, x, row, col)
            weights.append(d)
            total_weight = total_weight + d
            col = col + 1
        row = row + 1
    
    # Normalize to probabilities (roulette wheel)
    roulette_wheel = []
    accumulate = 0.0
    i = 0
    while i < len(weights):
        if total_weight > 0:
            accumulate = accumulate + (weights[i] / total_weight)
        roulette_wheel.append(accumulate)
        i = i + 1
    
    def move_to():
        choice = random.random()
        i = 0
        while i < len(roulette_wheel):
            if choice <= roulette_wheel[i]:
                return i // GRID_SIZE, i % GRID_SIZE
            i = i + 1
        return 0, 0  # Fallback (should not happen)
    
    movers = []
    cell_pop = grid[y][x]
    p_idx = 0
    while p_idx < len(cell_pop):
        p = cell_pop[p_idx]
        p_type = population[p][1]
        
        # Use correct index order (y,x)
        sim_val = similarity[y][x]
        
        condition_move = False
        if p_type == 0 and sim_val < SIMILARITY_THRESHOLD:
            condition_move = True
        elif p_type == 1 and (1.0 - sim_val) < SIMILARITY_THRESHOLD:
            condition_move = True
            
        if condition_move:
            nr, nc = move_to()
            movers.append([p, nr, nc])
            
        p_idx = p_idx + 1
        
    return movers

In [116]:
def migrate(grid, population, similarity):
    movers = []
    r_max = len(grid)
    c_max = len(grid[0])
    
    # Collect all movers
    row = 0
    while row < r_max:
        col = 0
        while col < c_max:
            m = move(row, col, grid, population, similarity)
            # Append movers
            i = 0
            while i < len(m):
                movers.append(m[i])
                i = i + 1
            col = col + 1
        row = row + 1
    
    # Execute moves
    i = 0
    while i < len(movers):
        m = movers[i]
        p_id = m[0]  # This is the actual index in population array
        new_r = m[1]
        new_c = m[2]
        
        # Boundary checking
        if new_r < 0 or new_r >= r_max or new_c < 0 or new_c >= c_max:
            i = i + 1
            continue  # Skip invalid moves
        
        p = population[p_id]
        old_r = p[3]
        old_c = p[4]
        
        # Skip if moving to same cell
        if old_r == new_r and old_c == new_c:
            i = i + 1
            continue
        
        # Update population record
        p[3] = new_r
        p[4] = new_c
        
        # Remove from old cell
        old_cell = grid[old_r][old_c]
        j = 0
        found = False
        while j < len(old_cell) and not found:
            if old_cell[j] == p_id:
                # Remove element by shifting remaining elements
                k = j
                while k < len(old_cell) - 1:
                    old_cell[k] = old_cell[k + 1]
                    k = k + 1
                old_cell.pop()  # Remove last element
                found = True
            j = j + 1
        
        # Add to new cell
        grid[new_r][new_c].append(p_id)
        
        i = i + 1
    
    return len(movers)
    

In [117]:
## Take this on faith
def plot_grid(grid):
    plt.imshow(grid, vmin=0.0, vmax=1.0)
    plt.colorbar()
    plt.show()

In [118]:
def draw_grid(svg,grid,population):
    r_max = len(grid)
    c_max = len(grid[0])
    row = 0
    
    while row < r_max:
        col = 0
        while col < c_max:
            type_0 = 0
            for p in grid[row][col]:
                if population[p][1]==0:
                    type_0+=1
            if type_0 >0:     
                #print(type_0/len(grid[row][col]),end=" ")
                r_col = 255-int(255*(type_0/len(grid[row][col])))
                svg.addRect(row,col,f'rgb(255,{r_col},{r_col})')
            col = col + 1
        #print()
        row = row + 1
                                            
    

In [119]:

# Initialize data structures
grid = []
similarity = []
distance = []
population = []

# Initialize grid and population
init_grid(grid, similarity, distance, GRID_SIZE)
init_population(population, grid)
calc_similarity(grid, population, similarity)

# Plot initial similarity
# print("Initial similarity grid:")
# plot_grid(similarity)

# Run simulation for 600 steps
# print(f"Running simulation for {MAX_ITERATIONS} steps...")

    
    # Optional: Print progress every 100 steps
    # if (t + 1) % 100 == 0:
    #     print(f"Completed step {t + 1}/MAX_ITERATIONS")

# Plot final similarity
# print(f"\n Iteration {t} similarity grid:")
# plot_grid(similarity)





In [120]:
svg = svg_canvas.SVGCanvas(500,500,GRID_SIZE)
# 1. Create the output ONCE and give it a permanent ID
output_display = display(HTML("<div id='sim-container'>Loading...</div>"), display_id=True)
for t in range(MAX_ITERATIONS):
    migrate(grid, population, similarity)
    calc_similarity(grid, population, similarity)
    svg.clear()
    draw_grid(svg,grid,population)
    new_svg = svg.getCanvas()
    output_display.update(HTML(f"<div id='sim-container'>{new_svg}</div>"))
    time.sleep(0.1)